# Lab 2 - Redes Neurais

Classificador: **MLP (MultiLayer Perceptron)**.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime

from lab2_utils import avaliar_modelo, logar_mlflow, iniciar_run

## Carregamento dos Dados Processados

In [2]:
data_dir = './dados_processados'

X_train = pd.read_parquet(f'{data_dir}/X_train.parquet')
y_train = pd.read_parquet(f'{data_dir}/y_train.parquet').values.ravel()
X_test = pd.read_parquet(f'{data_dir}/X_test.parquet')
y_test = pd.read_parquet(f'{data_dir}/y_test.parquet').values.ravel()

mod_time = datetime.fromtimestamp(os.path.getmtime(f'{data_dir}/X_train.parquet'))
print(f'Dados carregados (gerados em {mod_time.strftime("%Y-%m-%d %H:%M")})')
print(f'  X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'  y_train: {y_train.shape}  |  y_test: {y_test.shape}')
print(f'  Features: {list(X_train.columns)}')
print(f'  Churn rate treino: {y_train.mean():.3f}  |  teste: {y_test.mean():.3f}')

Dados carregados (gerados em 2026-06-24 15:43)
  X_train: (440832, 11)  |  X_test: (64374, 11)
  y_train: (440832,)  |  y_test: (64374,)
  Features: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Contract Length', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard']
  Churn rate treino: 0.567  |  teste: 0.474


## Configuração do MLflow

O módulo `lab2_utils.py` centraliza toda a configuração. Ao importá-lo, ele:

1. **Inicia o servidor MLflow automaticamente** (se não estiver rodando)
2. Configura tracking via HTTP (`http://127.0.0.1:5000`)
3. Garante runs idempotentes (re-executar substitui o run anterior)

A UI fica disponível em **http://127.0.0.1:5000** — basta abrir no navegador.

In [3]:
# Funções avaliar_modelo() e logar_mlflow() vêm do lab2_utils.py
# Nada a definir aqui — célula mantida para compatibilidade de estrutura.
print("MLflow configurado via lab2_utils.py")
print(f"  Experimento: Lab2-Churn")
print(f"  Tracking URI: sqlite:///mlflow.db")

MLflow configurado via lab2_utils.py
  Experimento: Lab2-Churn
  Tracking URI: sqlite:///mlflow.db


## Exemplo de Uso — Padrão MLflow

A função `iniciar_run()` cria o run com tags padronizadas (notebook de origem, modelo).
O `run_name` é fixo (nome do modelo) — cada execução gera um novo run diferenciado pelo horário e features na UI.

```python
params = {'modelo': 'MLP', 'hidden_layer_sizes': '(64, 32)', ...}
with iniciar_run("MLP", notebook="2B", params=params):
    model.fit(...)
    metricas = avaliar_modelo('MLP', y_test, y_pred)
    logar_mlflow(metricas, params, artefatos=[loss_path])
```

In [ ]:
# ─── MLP (v4) ──────────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier

hidden = (256, 128, 64)
params = {
    'modelo': 'MLP',
    'hidden_layer_sizes': str(hidden),
    'learning_rate_init': 0.0005,
    'max_iter': 300,
    'activation': 'relu',
    'solver': 'adam',
    'early_stopping': True,
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("MLP", notebook="2B", params=params):
    model = MLPClassifier(
        hidden_layer_sizes=hidden,
        learning_rate_init=0.0005,
        max_iter=300,
        activation='relu',
        solver='adam',
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    metricas = avaliar_modelo('MLP', y_test, y_pred)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(model.loss_curve_, label='Treino')
    if hasattr(model, 'validation_scores_'):
        ax.plot(model.validation_scores_, label='Validação')
    ax.set_title('Loss por Época — MLP')
    ax.set_xlabel('Época')
    ax.set_ylabel('Loss / Score')
    ax.legend()
    plt.tight_layout()
    loss_path = '../relatorio/imagens/2b_rn_loss_por_epoca.png'
    plt.savefig(loss_path, dpi=150, bbox_inches='tight')
    plt.show()

    logar_mlflow(metricas, params, artefatos=[loss_path])